# QED-DFT for Vibrational Polariton Infrared Spectrum

**Created by:** Khang Luong  
**Department of Chemistry, Brandeis University**
#### **Reference:** *[Insert Reference Here Later]*
--- 
## Overview

In previous lessons, we explored how an optical cavity creates Excited State Polaritons by coupling electronic excitations to quantized photons. However, we assumed the molecule's ground state remained unchanged by the cavity (the "Gas-Phase Ground State" approximation).

In this lesson, we moved beyond this approximation to explore Ground State QED-Kohn-Sham (QED-KS) theory. Here, the cavity's vacuum field is included self-consistently during the SCF cycle, meaning the cavity acutally reshapes the electron density in the molecule.

By the end of this tutorial, you will be able to:

1. **Part 1:** Ground State QED-DFT
2. **Part 2:** Analytical Gradients and Hessians
3. **Part 3:** IR Intensity and Spectra
4. **Part 4:** Vibrational-Photonic Hybridization
5. **Part 5:** Frequency Sweeps and Photon Character
6. **Part 6:** Capstone Exercise

---
# Environment Setup
### Option A: Running in Google Colab
If you opened this notebook in Google Colab, the enviroment configuration is handled automatically. 
* Run the **Setup Cell** below. It will clone the repository, setup the directory structure, and install all dependencies.

In [ ]:
import sys
import os

if 'google.colab' in sys.modules:
    print("Setting up Colab environment...")
    
    # Clone the repository to access local modules and data files
    !git clone https://github.com/cc-ats/molecular-polariton-modeling.git
    
    # Move into the repository directory
    %cd molecular-polariton-modeling
    
    # Install required packages
    !pip install -r requirements.txt
    
    print("Colab environment set up successfully!")
else:
    print("Running locally. No setup needed.")

### Option B: Running on Local Machine
If you are running it on your local computer, please refer to Lesson 1 for specific instruction on how to configure the environment.

In [ ]:
import warnings

warnings.filterwarnings('ignore')

# Color for printing
RED    = "\033[91m"
GREEN  = "\033[92m"
YELLOW = "\033[93m"
BLUE   = "\033[94m"
RESET  = "\033[0m"

# Unit Conversion:
HF_TO_EV = 27.2114
EV_TO_HF = 1 / HF_TO_EV

---
# ✔️ Part 1: Ground State QED-DFT (QED-KS)
In this part, we will calculate the ground state energy of a simple polar molecule and observe how the electron density responds to the cavity.

## 1.1 Coherent States vs. Number States 
### A.  Number States
The effective ground state potential energy in the **Number (Fock) state** basis is expressed as
$$
E = E_0 + \frac{1}{2}\sum_\alpha(p^2_\alpha + \omega_\alpha^2q_\alpha^2) + \sum_\alpha\left[\omega_\alpha q_\alpha(\mu_{\text{n},\alpha}+\mathrm{P} \cdot \mathrm{d}_\alpha)+\mu_{\text{n},\alpha}(\mathrm{P} \cdot \mathrm{d}_\alpha)+\frac{1}{2}\mu_{\text{n},\alpha}^2+\frac{1}{2}(\mathrm{\mathcal{Q}}+\mathrm{P}\cdot\mathrm{\tilde{\Delta}})\cdot\mathrm{P}\right]
$$
where $\mathrm{P}$ is the one particle density matrix and $\tilde{\Delta}_{\mu\nu, \lambda\sigma} = \Delta_{\mu\nu, \lambda\sigma} - \frac{1}{2} \Delta_{\mu\sigma, \lambda\nu}$, which also includes the exchange type dipolse self energy (DSE) contribution for restricted close-shell KS-DFT solutions. The electronic dipole, dipole-dipole, and quadrupole is therefore
$$
(d_{\alpha})_{\mu\nu}  = - e \sum_{x} \lambda_{\alpha, x} \bra{\phi_{\mu}} x \ket{\phi_\nu}
$$
$$
\Delta_{\mu\nu, \lambda\sigma} = \sum_{\alpha} (d_{\alpha})_{\mu\nu} (d_{\alpha})_{\lambda\sigma}
$$
$$
\mathcal{Q}_{\mu\nu} = e^{2} \sum_{\alpha} \sum_{xy} \lambda_{\alpha, x} \lambda_{\alpha, y} \bra{\phi_{\mu}} xy \ket{\phi_{\nu}}
$$
in the atomic orbital $(\phi)$ basis.

### B. Coherent States
The effective ground state potential energy in the **coherent state** is
$$
E' = E_0 + \sum_\alpha \hbar \omega_\alpha(n_\alpha +\frac{1}{2}) + \frac{1}{2}(\mathcal{Q}+\mathrm{P} \cdot \mathrm{\Delta}_k) \cdot \mathrm{P}
$$
where the Coulombic-type DSE contributions cancel each other, and the exhange-type DSE contribution involves $(\Delta_k)_{\mu\nu,\lambda\sigma}=-\frac{1}{2}\Delta_{\mu\sigma,\lambda\nu}$.


### C. The library provides two classes for QED-KS:
* `polariton_cs`: Uses Photon Coherent States. This is the standard choice for ground-state chemistry as it minimizes the zero-point energy of the field.
* `polariton_ns`: Uses Photon Number States (Fock states).

In [ ]:
import numpy as np
from pyscf import gto, lib, scf
from vibrational.qed_ks import polariton_cs
import matplotlib.pyplot as plt

First, we need to do a ground state calculation:

In [ ]:
mol = gto.M(
    atom = """
    H    0. 0. -0.459
    F    0. 0.  0.459
""",
    basis='6-311++g**',
)

mf_gas = scf.RKS(mol)
mf_gas.xc = 'pbe0'
# Here we use prune to reduce the number of grid points where they are uncesssary
mf_gas.grids.prune = True
e_gas = mf_gas.kernel()

# Extract dipole moments
dip_gas = mf_gas.dip_moment()

## 1.2 The "Self-Consistent" Polarization
When we run a `polariton_cs` calculation, the code adds the DSE terms to the Fock matrix. As the SCF converges, the electrons "feel" the vacuum field and polarize to find the new energy minimum


In [ ]:
mf_qed = polariton_cs(mol) # Coherent State
mf_qed.xc = 'pbe0'

# Define a couplign vector along the Z-axis
coupling = [0.0, 0.0, 0.03]
mf_qed.get_multipole_matrix(coupling)
e_qed = mf_qed.kernel()
dip_qed = mf_qed.dip_moment()

Finally, we print the results

In [ ]:
print(f"{GREEN}--------------------------------------------------------{RESET}")
print(f"{'Property':<20} | {'Gas Phase':<10} | {'QED-KS':<10}")
print(f"{GREEN}--------------------------------------------------------{RESET}")
print(f"{'Total Energy (au)':<20} | {e_gas:<10.6f} | {e_qed:<10.6f}")
print(f"{'Dipole Z (Debye)':<20} | {dip_gas[2]:<10.4f} | {dip_qed[2]:<10.4f}")
print(f"{GREEN}--------------------------------------------------------{RESET}")
print(f"DSE Quadrupole Energy: {mf_qed.scf_summary['equad']:.6f} a.u.")
print(f"DSE Exchange Energy:   {mf_qed.scf_summary['edse_k']:.6f} a.u.")
print(f"Energy Shift (eV):     {(e_qed - e_gas)*HF_TO_EV:.4f} eV")

**What to notice:**
* Here, we can see that the QED-KS energy is higher than the gas-phase energy. This is due to the Dipole Self-Energy (DSE) terms, which are always positive and represent the energy cost of the molecule displacing in the vacuum field. 

* The Dipole Z value is also changing even though the nuclei hasn't moved yet. This is because the electron density has shifted in response to the vacuum fluctuations, changing the permanent dipole moment.

* Finally, the energy shift is split into "Quadrupole" part (from the $A^2$ term) and an "Exchange" part (from the electronic cross terms).

We just saw mathematically that the Z-axis dipole moment changed. Let's look at exactly what the cavity vacuum field did to the electrons to cause this. Below, we calculate the electron density matrix for both the gas-phase and the cavity-coupled molecule. By taking the difference ($\Delta P = P_{QED} - P_{gas}$) and plotting it in 3D, we can see exactly where the electrons were pushed and pulled by the cavity's Dipole Self-Energy.

In [ ]:
import py3Dmol
from pyscf.tools import cubegen

print(f"{BLUE}--- Generating 3D Density Difference ---{RESET}")

# Get both density matrices
dm_gas = mf_gas.make_rdm1()
dm_qed = mf_qed.make_rdm1()

# Calculate the difference (QED effect)
dm_diff = dm_qed - dm_gas

# Generate the .cube file for the difference density
cubegen.density(mol, 'density_diff.cube', dm_diff)

# Read the cube data
with open('density_diff.cube', 'r') as f:
    cube_data = f.read()

# Render the interactive 3D viewer
view = py3Dmol.view(width=600, height=400)
view.addModel(cube_data, 'cube')

# Set the molecular representation
view.setStyle({'stick': {'radius': 0.1}, 'sphere': {'scale': 0.2}})

# Add the volumetric data
# Red shows where electron density DECREASED
view.addVolumetricData(cube_data, "cube", {'isoval': -0.00005, 'color': 'red', 'opacity': 0.75})
# Blue shows where electron density INCREASED
view.addVolumetricData(cube_data, "cube", {'isoval': 0.00005, 'color': 'blue', 'opacity': 0.75})

view.zoomTo()
view.show()

---
# ✔️ Part 2: Analytical Gradients
We saw the that the cavity "polarized" the molecule's electron density. But what about the nuclei?

If the electrons shift, the forces acting on the atoms will also change. This means that the equilibrium geometry of the molecule can actually be modified by the vaccum field of the cavity.

## 2.1 The Gradient Class
To calculate these forces, we use the gradient method. For QED-KS, this method calculates the derivative of the effective potential energy in the Fock state basis with respect to the nuclear coordinates:
$$
F_i = \frac{\partial E}{\partial R_i}= \frac{\partial E_0}{\partial R_i} + \frac{1}{2} \left(\frac{\partial\mathcal{Q}}{\partial R_i}+\mathrm{P} \cdot \frac{\partial\tilde{\mathrm{\Delta}}}{R_i}\right) \cdot \mathrm{P}
+
\sum_{\alpha} \left[ (\omega_\alpha q_\alpha + \mu_{\alpha} ) \frac{\partial \mu_{\rm{n},\alpha}}{\partial R_i} + ( \omega_\alpha q_\alpha + \mu_{\rm{n},\alpha}) \mathrm{P} \cdot \frac{\partial \mathrm{d}_\alpha}{\partial R_i} \right]
$$
In the library, the `polariton_cs` object has an intergrated `Gradients` class that handles the complex derivatives of the $A^2$ term and the DSE.

1. Setup the QED-KS object and Calculate Forces

In [ ]:
g_qed = mf_qed.Gradients()
grad_qed = g_qed.kernel()
forces_qed = -grad_qed

2. Calculate Gas-Phase Forces for comparison

In [ ]:
g_gas = mf_gas.Gradients()
forces_gas = -g_gas.kernel()

3. Results Analysis

In [ ]:
print(f"{YELLOW}--------------------------------------------------------{RESET}")
print(f"{'Atom':<10} | {'Gas Force Z':<15} | {'QED Force Z':<15}")
print(f"{YELLOW}--------------------------------------------------------{RESET}")
for i, atom in enumerate(['H', 'F']):
    print(f"{atom:<10} | {forces_gas[i,2]:<15.6f} | {forces_qed[i,2]:<15.6f}")
print(f"{YELLOW}--------------------------------------------------------{RESET}")

**What to notice:**
* In the gas phase, the forces should be nearly zero because the molecule is at its equilibirum distance.
* However, inside the cavity, the forces should become non zero:
    * If the force on H is positve and F is negative, the cavity is trying to contract the bond.
    * If they are both positve of negative, the cavity is trying stretch the bond
* The cavity quantum field effectively acts like an "external" field that couples to the molecule's permanent and induced dipoles, physically tugging on the atoms.

--- 
# ✔️ Part 3: Vibrational Analysis (Frequencies)
If the cavity changes forces on atoms, it also changes the stiffness of the chemical bonds. We can measure this by calculating the vibrational frequencies. 

## 3.1 The QED Hessian
To find frequencies, we need the Hessian matrix (the matrix of second derivatives of the energy). In QED-DFT, the Hessian includes the curvature of the electronic energy and the curvature added by the DSE.
$$
\frac{\partial^2E'}{\partial R_i \partial q_\alpha}=\frac{\partial^2E'}{\partial q_\alpha \partial R_i}=\omega_\alpha \frac{\partial(\mathrm{d}_\alpha \cdot \mathrm{P})}{\partial R_i}
$$
$$
\frac{\partial^2 E'}{\partial q_\alpha \partial q_\beta}=\omega^2_\alpha \delta_{\alpha \beta}
$$

1. We will use the `Hessian` class and PySCF's `thermo` module to perform the harmonic analysis. 

In [ ]:
from pyscf.hessian import thermo
from qed.utils import print_matrix

hessobj_gas = mf_gas.Hessian()
h_gas = hessobj_gas.kernel()
# View Hessian matrix
print_matrix('Hessian Matrix:', h_gas, 5, 1)

# thermo.harmonic_analysis converts the Hessian into wavenumbers (cm-1)
results_gas = thermo.harmonic_analysis(mol, h_gas)
freq_gas = results_gas['freq_wavenumber'][0] # Take only the vibrational mode

2. Calculate the QED-KS Hessian & Frequencies

In [ ]:
hessobj_qed = mf_qed.Hessian()
h_qed = hessobj_qed.kernel()
print_matrix('Hessian Matrix:', h_qed, 5, 1)
results_qed = thermo.harmonic_analysis(mol, h_qed)
freq_qed = results_qed['freq_wavenumber'][0]

3. Results Analysis

In [ ]:
print(f"{RED}--------------------------------------------------------{RESET}")
print(f"{'State':<15} | {'Frequency (cm-1)':<20}")
print(f"{RED}--------------------------------------------------------{RESET}")
print(f"{'Gas Phase':<15} | {freq_gas:<20.2f}")
print(f"{'Inside Cavity':<15} | {freq_qed:<20.2f}")
print(f"{RED}--------------------------------------------------------{RESET}")
print(f"Frequency Shift: {freq_qed - freq_gas:.2f} cm-1")

**What to notice:**
* You should see a shift in the vibrational frequency.
    * If the frequency increases, the cavity has "stiffened" the bond.
    * If it decreases, the cavity has "softened" the bond.
* Typcially, for a bond oriented along the cavity field, the DSE term adds a parabolic energy penalty to any displacement that changes the dipole. This usually leads to a blue-shift (higher frequency).
* This demonstrates that simply putting a molecule in a dark cavity can change its fundamental vibrational properties, even withou external light.

## 🏋️ Excercise: The Isotropic Detuning
In vibrational strong coupling (VSC), changing the mass of an atom shifts its bare vibrational frequency, which can immediately knock it out of resonance with the cavity.
* **The Task:** Create a new molecule object where you use a heavier atom. Re-calculate the gas-phase Hessian and QED-KS Hessian.
* **Analysis:** How much does the asymmetric stretch frequency shift? If your cavity is still perfectly tuned to 2350 cm-1, is the new molecule still in the strong coupling regime or has the detuning destroyed the hybridization?

In [ ]:
# TODO: Change to a different system. 

---
# ✔️ Part 4: IR Spectroscopy and Vibrational Polaritons
In this final part, we simulate the Infrared (IR) spectrum. In a cavity, a vibration doesn't just shift, it can hybridize with the cavity photon to create two new states: the Lower Polariton (LP) and Upper Polariton (UP). This is called Vibrational Strong Coupling (VSC).

## 4.1 Transition Dipoles and Intensities
To get the spectrum, we need the IR intensity (I), which is proportional to the square of the dipole derivatives

We will simulate CO2 at the TPSSH/def2-tzvp level. We'll tune the cavity frequency near the asymetric stretch of CO2 ($\approx 2350 cm^-1$) and observe the Rabi Splitting

In [ ]:
from vibrational.vibrational_spectra import get_dipole_dev, infrared, fit_val
from vibrational.qed_thermo import get_g1_d1, harmonic_analysis

mol = gto.M(
    atom="""
    O    0. 0. 0.386715
    C    0. 0. 1.550000
    O    0. 0. 2.713285
""",
    basis='def2-tzvp',
    verbose=0,
)

mf = polariton_cs(mol)
mf.xc = 'tpssh'
mf.grids.prune = False
coupling = [0.0, 0.0, 0.03]
mf.get_multipole_matrix(coupling)
mf.kernel();

Now, we get the Matter Hessian matrix.

In [ ]:
hessobj = mf.Hessian()
h_matter = hessobj.kernel()
# Standard analysis to get normal modes
res_matter = thermo.harmonic_analysis(mol, h_matter)
print('freq_au:', res_matter['freq_au'])
print('freq_wavenumber:', res_matter['freq_wavenumber'])
print('force_const_dyne:', res_matter['force_const_dyne'])
print('reduced_mass:', res_matter['reduced_mass'])

dip_dev = get_dipole_dev(mf, hessobj)
print_matrix('dip_dev:', dip_dev)
sir = infrared(dip_dev, res_matter['norm_mode'])
print('infrared intensity:', sir)

**Bonus:** Visualizing Molecular Vibration

In [ ]:
coords_ang = mol.atom_coords() * 0.529177 
elements = [mol.atom_symbol(i) for i in range(mol.natm)]

mode_idx = -1 
displacements = res_matter['norm_mode'][mode_idx] 

frames = 20
xyz_string = ""
for t in range(frames):
    phase = np.sin(2 * np.pi * t / frames)
    
    scale_factor = 0.5 
    current_coords = coords_ang + (displacements * phase * scale_factor) # Scale the displacement so it is easy to see (exaggerated for visuals)
    
    xyz_string += f"{mol.natm}\nFrame {t}\n"
    for i in range(mol.natm):
        xyz_string += f"{elements[i]} {current_coords[i,0]:.4f} {current_coords[i,1]:.4f} {current_coords[i,2]:.4f}\n"

view = py3Dmol.view(width=400, height=300)
view.addModelsAsFrames(xyz_string, "xyz")
view.setStyle({'stick': {'radius': 0.15}, 'sphere': {'scale': 0.25}})
view.animate({'loop': 'forward', 'step': 1}) 
view.zoomTo()
view.show()

Next, we perform a Hybrid Analysis (Matter + Photon)

In [ ]:
# Tune photon frequency near the asymetric stretch (~2350 cm-1)
cm_to_au = np.sqrt(res_matter['force_const_au'][-1]) / res_matter['freq_wavenumber'][-1]
cavity_freq_au = 2350 * cm_to_au
g1 = get_g1_d1(mf, cavity_freq_au, hessobj) # Coupling strengths between modes
print_matrix('g1:', g1, 5, 5)

Plotting with Lorentzian Broadening

In [ ]:
Lorentzian_broadening = 20  # cm-1
photon_freq_cm = np.array([2348, 2373, 2399, 2325, 2340])
photon_freq = photon_freq_cm * cm_to_au  # Convert cm-1 to Hartree-like units

spectra = []
photon_chars = []

for freq_cm, freq in zip(photon_freq_cm, photon_freq):
    g1 = get_g1_d1(mf, freq, hessobj)
    res_hybrid = harmonic_analysis(
        mol,
        res_matter['force_const_au'],
        res_matter['norm_mode'],
        g1,
        freq,
    )

    intensities = infrared(dip_dev, res_hybrid['norm_mode'])
    ix, iy = fit_val(res_hybrid['freq_wavenumber'], intensities, Lorentzian_broadening)
    spectra.append((freq_cm, ix, iy))

    photon_mode = res_hybrid['total_mode'][-1, :]
    photon_chars.append(np.max(photon_mode**2) * 100)

fig = plt.figure(figsize=(8, 9))
ax1 = fig.add_subplot(211)
for freq_cm, ix, iy in spectra:
    ax1.plot(ix, iy, label=f'Photon freq: {freq_cm:.0f} cm^-1', alpha=0.9)
ax1.set_xticks(np.arange(2200, 2600, 100))
ax1.set_yticks([0, 400, 800, 1200])
ax1.set_xlabel('Cavity Frequency (cm$^{-1}$)', fontsize=16)
ax1.set_ylabel('Intensity', fontsize=14)
ax1.set_ylim(0, 650)
ax1.set_xlim(2200, 2550)
ax1.set_title('Hybrid IR Spectra for Different Photon Frequencies', fontsize=16)
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

ax2 = fig.add_subplot(212)
ax2.bar(photon_freq_cm, photon_chars, width=20, color='C1', alpha=0.8)
for x, y in zip(photon_freq_cm, photon_chars):
    ax2.text(x, y + 2, f'{y:.1f}%', ha='center', va='bottom', fontsize=10)
ax2.set_xlabel('Photon Frequency (cm$^{-1}$)', fontsize=14)
ax2.set_ylabel('Max Photon Character (%)', fontsize=14)
ax2.set_ylim(0, 105)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


## 🏋️ Excercise: THe Polarization Geometry
The light-matter coupling strength is dictated by a dot product: $\vec{\lambda} \cdot \vec{\mu}$.

* **The Task:** In your current setup, CO$_2$ is aligned along the Z-axis, and your cavity coupling vector is [0.0, 0.0, 0.03]. Change the cavity coupling vector to be purely orthogonal to the molecule: [0.03, 0.0, 0.0]. Re-run the IR spectrum calculation.

* **Analysis:** What happens to the Rabi splitting? Why does the spectrum revert to looking exactly like the bare, uncoupled gas-phase molecule?

In [ ]:
# TODO: Change the cavity coupling vector and rerun the IR spectrum calculation

---
# ✔️ Part 5: Frequency Sweeps and Photon Character
To observe the Vibrational Polartion effects properly, we usually "sweep" the cavity frequency across the molecular resonance. This creates an Avoided Crossing diagram

## 5.1 What is "Photon Character"?
The `harmonic_analysis` from `vibration` package returns a `total_mode` array. For each hybrid state(LP, UP, etc.), this array contains the mixing coeficient:
* Vibrational components: the first $3N$ elements. 
* Photon components: The last element (for single-mode cavity).

1. Setup CO2 and `mf_qed` like in the previous parts

In [ ]:
# TODO: based on the instruction above, setup mf_qed for CO2
# If you still using the mf_qed for HF, the following part won't work
mf_qed = polariton_cs(mol)
mf_qed.xc = 'tpssh'
mf_qed.get_multipole_matrix(coupling)
mf_qed.kernel();


2. Extract Photon Character and Store for plotting (this part might take a while)

In [ ]:
sweep_freqs_cm = np.linspace(2200, 2500, 11)
all_results = []

# Pre-calculate the "Matter" parts once to save time
hessobj = mf_qed.Hessian()
h_matter = hessobj.kernel()
res_matter = thermo.harmonic_analysis(mol, h_matter)
div_dev = get_dipole_dev(mf_qed, hessobj)

print(f"{'Cavity Freq':<15} | {'LP Freq':<10} | {'UP Freq':<10} | {'UP Photon %':<10}")
print(f"{BLUE}--------------------------------------------------------{RESET}")

for f_cm in sweep_freqs_cm:
    f_au = f_cm / 219474.63

    g1 = get_g1_d1(mf_qed, f_au, hessobj)
    res_hybrid = harmonic_analysis(mol, res_matter['force_const_au'], res_matter['norm_mode'], g1, f_au)

    # total_mode is (N_modes, N_vars). The last column is the photon variable
    # We look at the states closest to our resonance
    photon_coeffs = res_hybrid['total_mode'][-1, :]
    photon_character = photon_coeffs**2

    all_results.append({
        'cav_freq': f_cm,
        'hybrid_freqs': res_hybrid['freq_wavenumber'],
        'photon_char': photon_character,
        'intensities': infrared(dip_dev, res_hybrid['norm_mode'])
    })
    print(f"{f_cm:<15.1f} | {res_hybrid['freq_wavenumber'][-2]:<10.1f} | "
        f"{res_hybrid['freq_wavenumber'][-1]:<10.1f} | {photon_character[-1]*100:<10.1f}%")

4. Plotting the Frequency Sweep

In [ ]:
cav_freqs = np.array([res['cav_freq'] for res in all_results])
# Track the two hybrid state (LP and UP)
lp_freqs = np.array([res['hybrid_freqs'][-2] for res in all_results])
up_freqs = np.array([res['hybrid_freqs'][-1] for res in all_results])
# Track the photon character
up_photon_char = np.array([res['photon_char'][-1] for res in all_results])
lp_photon_char = np.array([res['photon_char'][-2] for res in all_results])

    Plot 1: Avoided Crossing and Photon Character

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 10), sharex=True)
plt.subplots_adjust(hspace=0.05) # Minimizes the gap between the two plots

# Top Panel: Avoided Crossing (Energies)
ax1.plot(cav_freqs, up_freqs, 'ro-', linewidth=2, markersize=4, label='Upper Polariton (UP)')
ax1.plot(cav_freqs, lp_freqs, 'bo-', linewidth=2, markersize=4, label='Lower Polariton (LP)')

ax1.set_ylabel('Hybrid Frequency (cm$^{-1}$)', fontsize=12)
ax1.set_title('Vibrational Strong Coupling: Avoided Crossing', fontsize=14)
ax1.grid(True, alpha=0.3)
ax1.legend(loc='lower right')

# Bottom Panel: Mixing Coefficients (Photon Character)
ax2.plot(cav_freqs, up_photon_char, 'r^-', linewidth=2, markersize=5, label='UP Photon %')
ax2.plot(cav_freqs, lp_photon_char, 'b^-', linewidth=2, markersize=5, label='LP Photon %')

ax2.set_xlabel('Cavity Frequency (cm$^{-1}$)', fontsize=12)
ax2.set_ylabel('Photon Character (Weight)', fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.legend(loc='center right')

plt.show()

---
## 🏆 Part 6: Capstone Challenge – Proving the Linear Splitting Law

In our frequency sweep (Part 5), we proved that tuning the cavity perfectly to the molecular resonance creates an Avoided Crossing, and the minimum energy gap between the Upper and Lower Polariton is the Rabi Splitting ($\Omega$). 

Theoretical QED predicts that for a single molecule, this splitting should scale perfectly linearly with the cavity coupling strength: $\Omega \approx 2 \lambda \omega_c |\langle 0 | \mu | 1 \rangle|$. 

Your final challenge is to prove this linear relationship from first principles.

### The Setup
Keep your CO$_2$ molecule exactly as it is, and lock your cavity frequency perfectly to the gas-phase asymmetric stretch resonance ($\omega_c \approx 2393$ cm$^{-1}$). 

### Your Tasks:
**1. The Coupling Sweep**
Instead of sweeping the cavity frequency, write a loop that sweeps the **coupling strength** ($\lambda$) from `0.00` to `0.05` a.u. (e.g., `lambdas = np.linspace(0.0, 0.05, 10)`).

Inside the loop, for each $\lambda$:
* Define your coupling vector along the Z-axis: `[0.0, 0.0, lam]`.
* Initialize the `polariton_cs` object and run the QED-KS ground state.
* Calculate the analytical Gradients and the QED Hessian.
* Run the `harmonic_analysis` to get the hybrid frequencies.

**2. Extracting the Splitting**
At each step, calculate the energy gap between the Upper Polariton and Lower Polariton (the Rabi Splitting) in cm$^{-1}$. Store these values in a list.

**3. Visualization**
Plot the Rabi Splitting (y-axis) against the Coupling Strength $\lambda$ (x-axis). 

### Analysis
Does your plot form a perfectly straight line starting from the origin? If so, you have successfully used an ab initio quantum-electrodynamical Hessian to derive the fundamental scaling law of Vibrational Strong Coupling!